# NestedSimPy — Inventory with Lookahead Decisions

[NestedSimPy](https://nestedsimpy.github.io/) — a periodic stock whose order decision is chosen by trying each candidate quantity in inner simulations (`env.decide`). This example uses `env.decide + set_inner_actions`.

See the [example page](https://nestedsimpy.github.io/official-parity/inventory-lookahead.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile inventory_lookahead_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
Periodic-review inventory with lookahead order decisions.

Covers:

- Lookahead actions: env.decide, set_inner_actions, outer_run_mode
- Scoring branches with recorded values (metric="cost")

Scenario:
  A stock faces Poisson demand each period. After demand, an order
  decision: the engine tries each candidate quantity in inner
  simulations (candidate first, the order-up-to rule afterwards) and
  executes the one with the lowest average cost. None is the
  order-up-to rule running as its own candidate.
"""


import numpy as np

RANDOM_SEED = 42
PERIODS = 8                # review periods in the real run
MEAN_DEMAND = 5.0          # Poisson demand per period
HOLD_COST = 1.0            # per unit on hand per period
SHORTAGE_COST = 9.0        # per unit short per period (lost sales)
ORDER_UP_TO = 10           # the base rule's target position
ACTIONS = [None, 0, 5, 10]  # None = the base rule's own decision
LOOKAHEAD = 4              # periods each inner branch runs
REPS = 4                   # inner branches per candidate

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples",
                                                "inventory_lookahead")


def base_policy(state):
    """Order up to ORDER_UP_TO on the inventory position."""
    position = state["net_inventory"] + int(state["pipeline"].level)
    return max(0, ORDER_UP_TO - position)


def periods(env, state):
    while True:
        yield env.timeout(1.0)
        landing = int(state["pipeline"].level)      # last period's order
        if landing:
            state["pipeline"].get(landing)
            state["net_inventory"] += landing
        state["net_inventory"] -= int(np.random.poisson(MEAN_DEMAND))
        on_hand = max(state["net_inventory"], 0)
        short = max(-state["net_inventory"], 0)
        state["net_inventory"] = on_hand            # lost sales

        period_cost = HOLD_COST * on_hand + SHORTAGE_COST * short
        env.record("cost", period_cost)             # scores the branches
        state["cost"] += period_cost

        # The decision: publishes a "review" event (the branch trigger),
        # the engine launches one inner simulation per (action, replication)
        # and this line returns the winning quantity -- or the branch's
        # own candidate inside a branch, or base_policy(state) where
        # nothing applies.
        order = yield from env.decide(base_policy, state)
        if order > 0:
            state["pipeline"].put(order)            # arrives next period


def run():
    np.random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    state = {
        "net_inventory": 10,
        "pipeline": NestedContainer(env, capacity=float("inf"), init=0,
                                    nested_id="pipeline"),
        "cost": 0.0,
    }
    env.process(periods(env, state))

    # No triggering configuration: with actions declared, the engine
    # branches on the event decide publishes.
    env.set_outer_stopping_condition(timeout=PERIODS + 0.5)
    env.set_inner_stopping_condition(relative_time=float(LOOKAHEAD))
    env.set_inner_repetitions(REPS)
    env.set_rng("independent")
    env.set_outer_seed(RANDOM_SEED)
    env.set_inner_actions(ACTIONS, metric="cost", outer_run_mode="rollout")
    env.set_output_options(out_dir=NESTED_OUTPUT_FOLDER, gzip_trace=False)
    env.nested_run()
    return state["cost"], env


if __name__ == "__main__":
    total, env = run()
    by_action = env.get_inner_results_by_action(metric="cost")
    print(f"total cost {total:.1f} over {PERIODS} periods "
          f"({len(by_action)} decisions)")
    first = min(by_action)
    for action, values in sorted(by_action[first].items(), key=lambda i: str(i[0])):
        valid = [v for v in values if v is not None]
        mean = sum(valid) / len(valid) if valid else float("nan")
        pick = " <- executed" if action == env.best_inner_action(
            trigger=first, metric="cost") else ""
        print(f"  first decision, action {action!r:6}: mean {mean:6.1f}{pick}")


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python inventory_lookahead_colab.py

## 3. Inspect the run

`OutputManager` reads the run folder and reports the trigger events, plots the outer trajectory, and exports the sample path.

In [ ]:
import glob, os
import pandas as pd

run = os.path.dirname(glob.glob("simpy_examples/inventory_lookahead/**/rollout", recursive=True)[0])

# One CSV per zoom level: per-action scores, the executed picks,
# one row per inner simulation, every decision inside every branch.
picks = pd.read_csv(f"{run}/rollout/picks.csv")
actions = pd.read_csv(f"{run}/rollout/actions.csv")
print(picks.to_string(index=False))
print()
print(actions.head(8).to_string(index=False))  # an empty action cell is the base policy
